# 病名為AI：Colab GPU 渲染

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f0909172434/The-Disease-Called-AI/blob/claude/compassionate-dijkstra-16o5bo/tools/colab_render.ipynb)

在 Colab 的 GPU 上把整支 MV（1920×1080、24 fps、215 秒、不含音訊）渲染成 MP4，再上傳到 GitHub Release。

**操作步驟**

1. **Runtime（執行階段）→ Change runtime type（變更執行階段類型）→ Hardware accelerator 選 GPU**。L4 或 A100 最好，T4 也可以。
2. 左側 **🔑 Secrets（密鑰）** → Add new secret：名稱 `GH_TOKEN`，值是 fine-grained GitHub PAT，Repository access **只選** `f0909172434/voicebank-cache`，權限 **Contents: Read and write**。再把 **Notebook access（筆記本存取）** 打開。用完可以隨時撤銷這組 token。
3. **Runtime → Run all（全部執行）**。（程式碼是公開 repo，不需要 token 就能下載。）
4. 選用：跑到 Google Drive 的授權視窗時按允許。已完成的 1 秒片段會存進 Drive；斷線後重跑時把 `COMMIT` 填成上次印出的 sha，就會從中斷處續跑。
5. 跑完後在自己的電腦執行 `tools/fetch_colab_render.sh mv-render-<sha>` 下載 MP4 與 render_log.json。

注意：Run all 途中有任何一格出現 `[STOP]` 就是刻意停下來（例如找不到 NVIDIA GPU），訊息裡會寫原因與處理方法。分頁請保持開著，長時間渲染時 Colab 閒置太久會斷線。

In [ ]:
#@title 1 · Parameters { display-mode: "form" }
BRANCH    = "claude/compassionate-dijkstra-16o5bo"  #@param {type:"string"}
COMMIT    = ""      #@param {type:"string"}
FROM      = 0       #@param {type:"number"}
TO        = 215     #@param {type:"number"}
WORKERS   = 0       #@param {type:"integer"}
CRF       = 12      #@param {type:"integer"}
USE_DRIVE = True    #@param {type:"boolean"}
#@markdown `COMMIT`: optional full 40-character sha to pin (empty = tip of `BRANCH`). `FROM`/`TO` in seconds. `WORKERS` 0 = pick automatically from vCPU / RAM / GPU memory.

# ---- advanced, normally leave alone ----
RELEASE_REPO       = "f0909172434/voicebank-cache"
REQUIRE_TOKEN      = True    # stop early, before the long render, if GH_TOKEN is missing or unusable
DRAFT_RELEASE      = False   # True: create the release as a draft (only repo writers see it; a published release of a public repo is world-readable)
DRY_RUN            = False   # True: the upload cell only prints what it would do
RETRIES            = 2       # --video resumes from finished chunks, so a crashed run is simply started again this many times
EXTRA_CHROME_FLAGS = ""      # added to every Chrome launch (space separated)
WORK               = "/content"
GITHUB_API         = "https://api.github.com"
GITHUB_UPLOADS     = "https://uploads.github.com"
DRIVE_ROOT         = "/content/drive/MyDrive"
GH_LIMIT           = 2 * 1024**3       # GitHub: one release asset must be smaller than 2 GiB ...
PART               = 1900 * 1024**2    # ... so a bigger video goes up as raw byte parts of this size (video.mp4.part001, ...)

In [ ]:
#@title 2 · Helpers (shared by the cells below; safe to re-run)
import hashlib, json, math, os, re, shutil, signal, subprocess, sys, threading, time, urllib.parse, urllib.request

REPO_URL   = "https://github.com/f0909172434/The-Disease-Called-AI.git"
REPO_DIR   = f"{WORK}/The-Disease-Called-AI"
FILM       = f"{REPO_DIR}/film"
OUT_DIR    = f"{WORK}/out"
STATE_FILE = f"{WORK}/.mv_state.json"
SUDO       = [] if (hasattr(os, "geteuid") and os.geteuid() == 0) else ["sudo"]
os.makedirs(OUT_DIR, exist_ok=True)


class Stop(RuntimeError):
    """An expected, explained failure: the message says what to do."""

def stop(*lines):
    raise Stop("\n\n[STOP] " + "\n".join(str(l) for l in lines).strip() + "\n")


# ---- state kept in a file, so a re-run of a single cell (or a kernel restart on the same VM) still knows what happened ----
def _load_state():
    try:
        with open(STATE_FILE) as f:
            return json.load(f)
    except Exception:
        return {}
S = _load_state()
def save():
    try:
        with open(STATE_FILE, "w") as f:
            json.dump(S, f, indent=1, default=str)
    except Exception as e:
        print("(could not save state:", e, ")")

def add_path(d):
    d = str(d)
    if d not in os.environ["PATH"].split(":"):
        os.environ["PATH"] = d + ":" + os.environ["PATH"]
    if d not in S.setdefault("path", []):
        S["path"].append(d); save()
for _d in list(S.get("path", [])):
    if os.path.isdir(_d): add_path(_d)


# ---- never print the token ----
_SECRETS = []
def redact(s):
    s = str(s)
    for t in _SECRETS:
        if t: s = s.replace(t, "***")
    return re.sub(r"(github_pat_|gh[pousr]_)[A-Za-z0-9_]{12,}", r"\1***", s)


# ---- run a command, stream its output live ----
class R:
    def __init__(self, rc, out): self.rc, self.out = rc, out
    def __bool__(self): return self.rc == 0

def sweep_chrome():
    """Chrome is started detached by puppeteer, so killing node's process group does not reach it: kill what is left of OUR Chrome."""
    c = S.get("chrome")
    if c: subprocess.run(["pkill", "-9", "-f", re.escape(c)], capture_output=True)

def _kill(p):
    for sig, patience in ((signal.SIGTERM, 5), (signal.SIGKILL, 0)):    # TERM first: node then closes its Chromes itself
        try: os.killpg(p.pid, sig)
        except Exception:
            try: p.send_signal(sig)
            except Exception: pass
        t = time.time()
        while patience and time.time() - t < patience and p.poll() is None: time.sleep(0.1)
        if p.poll() is not None: break
    sweep_chrome()

def sh(cmd, cwd=None, env=None, echo=True, check=True, timeout=None, on_line=None, logfile=None, what=None):
    """cmd: str (through bash) or list (exec). Streams stdout+stderr line by line (token-redacted); returns R(rc, out).
    on_line(line) -> True means 'handled, don't echo'. check=True: a non-zero exit raises Stop with the last lines."""
    shell = isinstance(cmd, str)
    p = subprocess.Popen(cmd, cwd=cwd, env=env or os.environ, shell=shell, executable="/bin/bash" if shell else None,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors="replace", bufsize=1, start_new_session=True)
    killer = threading.Timer(timeout, _kill, args=(p,)) if timeout else None
    if killer: killer.start()
    lines, lf = [], (open(logfile, "a") if logfile else None)
    try:
        for raw in p.stdout:
            line = redact(raw.rstrip("\n"))
            lines.append(line)
            if len(lines) > 4000: del lines[:1000]
            if lf: lf.write(line + "\n"); lf.flush()
            if on_line and on_line(line): continue
            if echo: print(line, flush=True)
        rc = p.wait()
    except BaseException:
        _kill(p); raise
    finally:
        if killer: killer.cancel()
        if lf: lf.close()
    if rc and check:
        shown = what or (cmd if shell else " ".join(map(str, cmd)))
        stop(f"`{shown[:140]}` failed (exit {rc}). Last output:", *lines[-25:])
    return R(rc, "\n".join(lines))


# ---- machine facts and the WORKERS pick ----
def detect_vcpu():
    n = os.cpu_count() or 1
    try: n = min(n, len(os.sched_getaffinity(0)))
    except Exception: pass
    try:                                               # cgroup v2 quota ("max 100000" or "200000 100000")
        q, per = open("/sys/fs/cgroup/cpu.max").read().split()
        if q != "max": n = min(n, max(1, math.ceil(int(q) / int(per))))
    except Exception: pass
    try:                                               # cgroup v1
        q = int(open("/sys/fs/cgroup/cpu/cpu.cfs_quota_us").read()); per = int(open("/sys/fs/cgroup/cpu/cpu.cfs_period_us").read())
        if q > 0: n = min(n, max(1, math.ceil(q / per)))
    except Exception: pass
    return n

def detect_ram_gb():
    try:
        kb = next(int(l.split()[1]) for l in open("/proc/meminfo") if l.startswith("MemTotal:"))
        gb = kb / 1048576
    except Exception:
        return 8.0
    for f in ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"):
        try:
            v = open(f).read().strip()
            if v != "max" and int(v) < (1 << 50): gb = min(gb, int(v) / 2**30)
        except Exception: pass
    return gb

def pick_workers(vcpu, gpu_mem_mib, ram_gb, cap=6):
    """One worker = one Chrome (main thread + GPU process) + one ffmpeg: ~1.3 vCPU, ~2.5 GB RAM, ~1.5 GB of VRAM."""
    by_cpu = max(1, int(vcpu * 0.75 + 0.5))
    by_gpu = max(1, int((gpu_mem_mib - 1024) // 1536))
    by_ram = max(1, int((ram_gb - 2) // 2.5))
    return max(1, min(by_cpu, by_gpu, by_ram, cap)), {"by_vcpu": by_cpu, "by_gpu_mem": by_gpu, "by_ram": by_ram, "cap": cap}


# ---- the page and the film ----
def check_gl_line(line):
    """render.mjs prints `GL: <flags> (<renderer>)` once the page is up. Anything but NVIDIA there = stop, never render on the CPU."""
    if line.startswith("GL: ") and not re.search(r"nvidia", line, re.I):
        stop("The page's WebGL is not on the NVIDIA GPU: " + line[:200], "Refusing to render on the CPU. Re-run the GPU probe cell.")

def n_frames(a, b, fps, dur):
    return max(0, min(round(dur * fps), round(b * fps)) - max(0, round(a * fps)))

def read_project():
    cfg = open(f"{FILM}/src/config.js").read()
    m = re.search(r"PROJECT\s*=\s*\{[^}]*?duration:\s*([\d.]+)[^}]*?fps:\s*(\d+)", cfg)
    return (float(m.group(1)), int(m.group(2))) if m else (215.0, 24)


# ---- small things ----
def fmt_dur(s):
    s = float(s)
    return f"{s:.0f} s" if s < 90 else f"{s / 60:.1f} min" if s < 5400 else f"{s / 3600:.2f} h"

def num(x):                                            # 55.0 -> "55", 0.5 -> "0.5" (render.mjs --from/--to take seconds)
    return str(int(x)) if float(x) == int(float(x)) else str(float(x))

def sha256_file(path, offset=0, length=None, bs=16 << 20):
    h, left = hashlib.sha256(), length
    with open(path, "rb") as f:
        f.seek(offset)
        while left is None or left > 0:
            b = f.read(bs if left is None else min(bs, left))
            if not b: break
            h.update(b)
            if left is not None: left -= len(b)
    return h.hexdigest()

def apt_install(pkgs):
    """apt-get install whatever of `pkgs` is missing; one unknown name must not sink the rest. Returns the names that failed."""
    q = subprocess.run(["dpkg-query", "-W", "-f", "${Package}\t${db:Status-Abbrev}\n", *pkgs], capture_output=True, text=True)
    have = {a for a, _, b in (l.partition("\t") for l in q.stdout.splitlines()) if b.strip().startswith("ii")}
    todo = [p for p in pkgs if p not in have]
    if not todo: return []
    env = {**os.environ, "DEBIAN_FRONTEND": "noninteractive"}
    if not S.get("apt_updated"):
        sh([*SUDO, "apt-get", "update", "-qq"], env=env, check=False, echo=False); S["apt_updated"] = True; save()
    base = [*SUDO, "apt-get", "install", "-y", "-qq", "--no-install-recommends"]
    if sh([*base, *todo], env=env, check=False, echo=False).rc == 0: return []
    failed = []
    for p in todo:
        if sh([*base, p], env=env, check=False, echo=False).rc != 0 and sh([*base, p + "t64"], env=env, check=False, echo=False).rc != 0:
            failed.append(p)
    return failed


# ---- GitHub REST ----
def get_token():
    """GH_TOKEN from Colab Secrets (google.colab.userdata); outside Colab, from the environment. Never printed."""
    tok = None
    try:
        from google.colab import userdata
    except ImportError:
        tok = os.environ.get("GH_TOKEN")
    else:
        try:
            tok = userdata.get("GH_TOKEN")
        except Exception as e:
            n = type(e).__name__
            if "NotFound" in n:
                stop("The secret GH_TOKEN does not exist.", "Left sidebar -> the key icon (Secrets) -> Add new secret: name GH_TOKEN, value = your fine-grained PAT.")
            if "Access" in n:
                stop("Notebook access to GH_TOKEN is off.", "Left sidebar -> Secrets -> switch on 'Notebook access' for GH_TOKEN, then run this cell again.")
            stop(f"Could not read the secret GH_TOKEN ({n}). Check Secrets and run this cell again.")
    if not tok:
        stop("GH_TOKEN is empty. Put a fine-grained PAT (only f0909172434/voicebank-cache, Contents: read and write) in Colab Secrets.")
    tok = tok.strip()
    if tok not in _SECRETS: _SECRETS.append(tok)
    return tok

def gh(method, path, token=None, tries=3, **kw):
    """One GitHub API call with a few retries on network errors and 5xx. `path` is relative to GITHUB_API unless it is a full URL."""
    import requests
    url = path if path.startswith("http") else GITHUB_API + path
    headers = {"Accept": "application/vnd.github+json", "X-GitHub-Api-Version": "2022-11-28", "User-Agent": "mv-colab-render", **kw.pop("headers", {})}
    if token: headers["Authorization"] = f"Bearer {token}"
    timeout = kw.pop("timeout", 60)
    for i in range(1, tries + 1):
        try:
            r = requests.request(method, url, headers=headers, timeout=timeout, **kw)
        except requests.RequestException as e:
            if i == tries: stop(f"GitHub {method} {urllib.parse.urlparse(url).path} failed: {redact(e)}")
            time.sleep(2 * i); continue
        if r.status_code >= 500 and i < tries:
            time.sleep(2 * i); continue
        return r

print("helpers ready.  work dir:", WORK, "| state:", {k: S[k] for k in ("short",) if k in S} or "(fresh)")

In [ ]:
#@title 3 · GPU check
if not shutil.which("nvidia-smi"):
    stop("No NVIDIA GPU in this runtime (nvidia-smi is missing).",
         "Runtime -> Change runtime type -> Hardware accelerator: GPU (L4 or A100 preferred, T4 is fine), then Run all again.")
sm = sh("nvidia-smi", check=False)
if sm.rc != 0:
    stop("nvidia-smi failed, so there is no usable GPU.",
         "Runtime -> Change runtime type -> GPU. If you already did: Runtime -> Disconnect and delete runtime, then connect again.")
q = sh("nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader,nounits", echo=False)
gname, driver, gmem = [x.strip() for x in q.out.strip().splitlines()[0].split(",")]
vcpu, ram = detect_vcpu(), detect_ram_gb()
auto, why = pick_workers(vcpu, int(float(gmem)), ram)
WORKERS_EFF = int(WORKERS) if int(WORKERS) > 0 else auto
free_gb = shutil.disk_usage(WORK).free / 2**30
S["gpu"] = {"name": gname, "driver": driver, "memory_mib": int(float(gmem)), "vcpu": vcpu, "ram_gb": round(ram, 1)}
S["workers"] = WORKERS_EFF; save()
print(f"\nGPU {gname} | driver {driver} | {int(float(gmem))} MiB | {vcpu} vCPU | {ram:.1f} GB RAM | {free_gb:.0f} GB free disk")
print(f"WORKERS = {WORKERS_EFF}" + ("  (your override)" if int(WORKERS) > 0 else f"  (auto: limits {why})"))
if free_gb < 8: print("WARNING: less than 8 GB free disk; the chunks and the final MP4 need about 4 GB.")
if vcpu <= 2: print("NOTE: only 2 vCPUs (free-tier T4). It works but is slow: the render is CPU-bound as much as GPU-bound; an L4/A100 runtime has more cores.")

In [ ]:
#@title 4 · Mount Google Drive (optional; asked now so the pop-up does not stall the run later)
DRIVE_OK = False
if not USE_DRIVE:
    print("USE_DRIVE = False: finished chunks stay on this VM only (a disconnect loses them).")
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_OK = os.path.isdir(DRIVE_ROOT)
    except Exception as e:
        print(f"Drive was not mounted ({type(e).__name__}: {redact(e)[:200]}).")
    print("Drive mounted: chunks will be kept in MyDrive/disease_called_ai/" if DRIVE_OK else
          "Continuing without Drive: finished chunks stay on this VM only (a disconnect loses them).")

In [ ]:
#@title 5 · Pre-flight: is GH_TOKEN usable? (so the upload does not fail after an hour of rendering)
if DRY_RUN or not REQUIRE_TOKEN:
    print("token pre-flight skipped (DRY_RUN or REQUIRE_TOKEN = False); the upload cell will still need GH_TOKEN.")
else:
    _t = get_token()
    r = gh("GET", f"/repos/{RELEASE_REPO}", _t)
    if r.status_code in (401, 403):
        stop(f"GitHub rejected GH_TOKEN (HTTP {r.status_code}): expired, revoked, or not allowed for {RELEASE_REPO}.",
             "Create a fine-grained PAT with Repository access = only that repo and Contents: Read and write, and put it in Secrets.")
    if r.status_code == 404:
        stop(f"GH_TOKEN cannot see {RELEASE_REPO} (HTTP 404): the PAT's repository access does not include it.")
    if r.status_code != 200:
        stop(f"Unexpected answer from GitHub (HTTP {r.status_code}) while checking {RELEASE_REPO}.")
    j = r.json()
    perms = j.get("permissions") or {}
    if perms and not perms.get("push"):
        stop(f"GH_TOKEN can read {RELEASE_REPO} but cannot write to it: give the PAT Contents: Read and write.")
    S["release_repo_private"] = bool(j.get("private")); save()
    print(f"GH_TOKEN OK for {RELEASE_REPO} (write access: {perms.get('push', 'not reported')}).")
    if not j.get("private") and not DRAFT_RELEASE:
        print(f"NOTE: {RELEASE_REPO} is a PUBLIC repo, so the release (and the video) will be downloadable by anyone with the link."
              " Set DRAFT_RELEASE = True in the parameters if that is not what you want.")

In [ ]:
#@title 6 · Fetch the code (shallow clone of the branch)
_genv = {**os.environ, "GIT_TERMINAL_PROMPT": "0", "GIT_ASKPASS": "true"}      # public repo: never ask for credentials
def git(*a, **kw): return sh(["git", *a], cwd=REPO_DIR, env=_genv, **kw)

os.makedirs(REPO_DIR, exist_ok=True)
if not os.path.isdir(f"{REPO_DIR}/.git"):
    sh(["git", "init", "-q", REPO_DIR], env=_genv)
git("remote", "remove", "origin", check=False, echo=False)
git("remote", "add", "origin", REPO_URL)

_pin = COMMIT.strip()
_ref = f"+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"
if _pin:
    if git("fetch", "-q", "--depth", "1", "origin", _pin, check=False, echo=False).rc == 0:
        git("checkout", "-q", "-f", "--detach", "FETCH_HEAD")
    else:       # a short sha cannot be fetched directly: take the branch history and look for it there
        git("fetch", "-q", "--depth", "1000", "origin", _ref, what=f"git fetch origin {BRANCH}")
        git("checkout", "-q", "-f", "--detach", _pin, what=f"git checkout {_pin}")
else:
    git("fetch", "-q", "--depth", "1", "origin", _ref, what=f"git fetch origin {BRANCH}")
    git("checkout", "-q", "-f", "--detach", f"origin/{BRANCH}")

SHA = git("rev-parse", "HEAD", echo=False).out.strip()
SHORT = SHA[:7]
TAG = f"mv-render-{SHORT}"
S.update(sha=SHA, short=SHORT, tag=TAG, branch=BRANCH, subject=git("log", "-1", "--format=%s", echo=False).out.strip()); save()
print(f"commit {SHA}\n       {git('log', '-1', '--format=%ci  %s', echo=False).out.strip()}")

_need = ["film/render.mjs", "film/studio.html", "film/package-lock.json", "film/vendor/p5.min.js", "film/vendor/p5.brush.js",
         "music/build/arrangement.json", "music/build/vocals.json"]
_missing = [f for f in _need if not os.path.exists(f"{REPO_DIR}/{f}")]
if _missing: stop("The checkout is missing files the renderer needs:", *_missing)
if "probe-gl" not in open(f"{FILM}/render.mjs").read():
    stop(f"film/render.mjs at {SHORT} has no --probe-gl flag (the GPU probe needs it).",
         "Commit and push film/render.mjs together with this notebook, or leave COMMIT empty to take the tip of the branch.")
print(f"\nrelease tag will be {TAG}")
print(f'To resume after a disconnect (finished chunks are reused), set COMMIT = "{SHA}" in the parameters.')

In [ ]:
#@title 7 · Install 1/4: system libraries, Node 22, npm packages
# puppeteer-core 25 (film/package.json) needs Node >= 22.12; Colab ships an older one, so Node comes from nodejs.org if needed.
CHROME_LIBS = ["ca-certificates", "fonts-liberation", "xz-utils", "xdg-utils", "libasound2", "libatk-bridge2.0-0", "libatk1.0-0", "libcairo2", "libcups2",
               "libdbus-1-3", "libdrm2", "libexpat1", "libgbm1", "libglib2.0-0", "libnspr4", "libnss3", "libpango-1.0-0", "libx11-6", "libxcb1",
               "libxcomposite1", "libxdamage1", "libxext6", "libxfixes3", "libxkbcommon0", "libxrandr2", "libxshmfence1"]
GL_LIBS = ["libvulkan1", "libegl1", "libgles2", "libgl1", "libglvnd0"]          # the loaders ANGLE talks to (the NVIDIA side comes in step 4/4)

_failed = apt_install(CHROME_LIBS + GL_LIBS)
print("system libraries: ok" if not _failed else f"system libraries: could not install {_failed} (the Chrome step re-checks what is really missing)")

NODE_MIN = (22, 12, 0)
def node_version():
    r = sh("node -v", echo=False, check=False)
    m = re.match(r"v(\d+)\.(\d+)\.(\d+)", r.out.strip())
    return tuple(int(x) for x in m.groups()) if m else None

nv = node_version()
if nv and nv >= NODE_MIN:
    print(f"node v{'.'.join(map(str, nv))}: ok")
else:
    print(f"node {nv and 'v' + '.'.join(map(str, nv))} is too old; installing Node 22 from nodejs.org ...")
    base = "https://nodejs.org/dist/latest-v22.x"
    sums = urllib.request.urlopen(f"{base}/SHASUMS256.txt", timeout=60).read().decode()
    m = re.search(r"^([0-9a-f]{64})\s+(node-v22\.\d+\.\d+-linux-x64\.tar\.xz)$", sums, re.M)
    if not m: stop("Could not find a linux-x64 Node 22 tarball in nodejs.org's SHASUMS256.txt.")
    want, tarball = m.groups()
    tgz = f"{WORK}/{tarball}"
    sh(["curl", "-fsSL", "--retry", "3", "-o", tgz, f"{base}/{tarball}"], echo=False)
    if sha256_file(tgz) != want: stop("Node tarball checksum mismatch; run this cell again.")
    sh(["tar", "-xJf", tgz, "-C", "/opt"], echo=False)
    add_path(f"/opt/{tarball[:-len('.tar.xz')]}/bin")
    nv = node_version()
    if not nv or nv < NODE_MIN: stop(f"Node install did not give a usable node (got {nv}). PATH={os.environ['PATH']}")
    print(f"node v{'.'.join(map(str, nv))}: installed to /opt/{tarball[:-len('.tar.xz')]}")
S["node"] = "v" + ".".join(map(str, nv)); save()

lock = sha256_file(f"{FILM}/package-lock.json")
if S.get("npm_lock") == lock and os.path.exists(f"{FILM}/node_modules/.bin/browsers"):
    print("npm packages: already installed for this lockfile")
else:
    if sh("npm ci --no-audit --no-fund --loglevel=error", cwd=FILM, check=False).rc != 0:
        print("npm ci failed; trying npm install ...")
        sh("npm install --no-audit --no-fund --loglevel=error", cwd=FILM)
    if not os.path.exists(f"{FILM}/node_modules/puppeteer-core/package.json"): stop("npm finished but puppeteer-core is not in film/node_modules.")
    S["npm_lock"] = lock; save()
    print("npm packages: installed (puppeteer-core)")

In [ ]:
#@title 8 · Install 2/4: Chrome (the build puppeteer-core drives)
def chrome_works(path):
    r = sh([path, "--version"], echo=False, check=False, timeout=60)
    return r.rc == 0 and "chrome" in r.out.lower()

def missing_libs(path):
    r = sh(["ldd", path], echo=False, check=False)
    return sorted({l.split()[0] for l in r.out.splitlines() if "not found" in l})

CHROME = S.get("chrome")
if CHROME and os.path.exists(CHROME) and chrome_works(CHROME):
    print("chrome already installed:", CHROME)
else:
    CHROME = None
    # 1) Chrome for Testing through @puppeteer/browsers (already in node_modules): the build puppeteer is tested against
    r = sh([f"{FILM}/node_modules/.bin/browsers", "install", "chrome@stable", "--path", f"{WORK}/chrome", "--format", "{{path}}"], check=False, echo=False, timeout=900)
    cand = next((l.strip() for l in reversed(r.out.splitlines()) if os.path.exists(l.strip())), "")
    if r.rc == 0 and cand:
        CHROME = cand
    else:
        print("@puppeteer/browsers failed (", r.out.strip()[-200:], ") -> trying Google's .deb")
        # 2) Google's own package (apt pulls every dependency)
        sh(f"curl -fsSL --retry 3 -o {WORK}/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb && "
           f"{' '.join(SUDO)} DEBIAN_FRONTEND=noninteractive apt-get install -y -qq {WORK}/chrome.deb", echo=False, timeout=900)
        CHROME = shutil.which("google-chrome") or "/usr/bin/google-chrome"
    # shared libraries Chrome still cannot find
    miss = missing_libs(CHROME)
    if miss:
        print("Chrome is missing shared libraries:", miss, "-> installing the usual set again")
        apt_install(CHROME_LIBS)
        miss = missing_libs(CHROME)
        if miss: stop("Chrome still cannot find these shared libraries:", *miss, "Install the matching apt packages and run this cell again.")
    if not chrome_works(CHROME): stop(f"{CHROME} does not start (--version failed).")
S["chrome"] = CHROME
S["chrome_version"] = sh([CHROME, "--version"], echo=False, check=False).out.strip()
save()
print(f"chrome: {S['chrome_version']}\n        {CHROME}")

In [ ]:
#@title 9 · Install 3/4: ffmpeg (render.mjs uses `-fps_mode`, which needs ffmpeg >= 5.1; Colab ships 4.4)
def ffmpeg_smoke():
    """The exact encoder options render.mjs uses for its chunks, on one second of test pattern."""
    t = f"{WORK}/ffmpeg_smoke.mp4"
    r = sh(["ffmpeg", "-y", "-loglevel", "error", "-f", "lavfi", "-i", "testsrc=duration=1:size=128x72:rate=24", "-c:v", "libx264", "-preset", "medium",
            "-crf", "12", "-profile:v", "high", "-pix_fmt", "yuv420p", "-r", "24", "-fps_mode", "cfr", "-video_track_timescale", "24000",
            "-threads", "2", "-movflags", "+faststart", "-f", "mp4", t], echo=False, check=False)
    return r.rc == 0 and os.path.exists(t) and os.path.getsize(t) > 0

def ffmpeg_line():
    return sh("ffmpeg -version | head -1", echo=False, check=False).out.strip()

if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    apt_install(["ffmpeg"])
if ffmpeg_smoke():
    print("ffmpeg ok:", ffmpeg_line())
else:
    print("the system ffmpeg cannot run render.mjs's options:", ffmpeg_line(), "\ninstalling a newer static build ...")
    done = False
    # 1) imageio-ffmpeg: a static ffmpeg inside a PyPI wheel (PyPI is always reachable from Colab)
    sh([sys.executable, "-m", "pip", "install", "-q", "imageio-ffmpeg"], check=False, echo=False)
    try:
        import imageio_ffmpeg
        os.makedirs("/opt/ffmpeg-bin", exist_ok=True)
        if os.path.lexists("/opt/ffmpeg-bin/ffmpeg"): os.remove("/opt/ffmpeg-bin/ffmpeg")
        os.symlink(imageio_ffmpeg.get_ffmpeg_exe(), "/opt/ffmpeg-bin/ffmpeg")
        add_path("/opt/ffmpeg-bin")
        done = ffmpeg_smoke()
    except Exception as e:
        print("imageio-ffmpeg route failed:", redact(e)[:200])
    # 2) static builds with a stable file name: BtbN's master build (GitHub), then johnvansickle.com's release build (both include ffprobe)
    for url in ("https://github.com/BtbN/FFmpeg-Builds/releases/download/latest/ffmpeg-master-latest-linux64-gpl.tar.xz",
                "https://johnvansickle.com/ffmpeg/releases/ffmpeg-release-amd64-static.tar.xz"):
        if done: break
        d = f"/opt/ffmpeg-static-{url.split('/')[2].split('.')[0]}"
        if sh(f"rm -rf {d} && mkdir -p {d} && curl -fsSL --retry 3 -o {WORK}/ffmpeg.tar.xz {url} && tar -xJf {WORK}/ffmpeg.tar.xz -C {d} --strip-components=1",
              echo=False, check=False, timeout=900).rc != 0:
            print("  download failed:", url.split("/")[2]); continue
        exe = sh(f"find {d} -type f -name ffmpeg | head -1", echo=False, check=False).out.strip()
        if exe:
            add_path(os.path.dirname(exe)); done = ffmpeg_smoke()
    if not done: stop("Could not get an ffmpeg that runs render.mjs's encoder options (-fps_mode cfr, libx264).")
    print("ffmpeg ok:", ffmpeg_line(), "->", shutil.which("ffmpeg"))
if not shutil.which("ffprobe"): stop("ffprobe is missing (it comes with ffmpeg); install the ffmpeg package.")
S["ffmpeg"] = ffmpeg_line(); S["ffmpeg_path"] = shutil.which("ffmpeg"); save()

In [ ]:
#@title 10 · Install 4/4: the NVIDIA graphics libraries ANGLE needs (only if Colab does not already have them)
# Colab's driver mounts the compute libraries (CUDA, NVML) but usually NOT the OpenGL/EGL/Vulkan ones. ANGLE reaches the GPU
# through libEGL_nvidia / libGLX_nvidia (+ the Vulkan ICD json), and those must be the exact version of the kernel driver.
DRIVER = S["gpu"]["driver"]
LIBDIR = "/usr/lib/x86_64-linux-gnu"
ICD_PATH = "/etc/vulkan/icd.d/nvidia_icd.json"
EGL_PATH = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"

def lib_dirs():
    return [LIBDIR, "/usr/lib64-nvidia", "/usr/local/nvidia/lib64", "/usr/local/nvidia/lib"] + [d for d in os.environ.get("LD_LIBRARY_PATH", "").split(":") if d]

def nv_gl_status():
    ld = sh("ldconfig -p", echo=False, check=False).out
    dirs = lib_dirs()
    def has(name): return name in ld or any(os.path.exists(f"{d}/{name}") for d in dirs)
    icd = [p for p in (ICD_PATH, "/usr/share/vulkan/icd.d/nvidia_icd.json", "/usr/local/share/vulkan/icd.d/nvidia_icd.json") if os.path.exists(p)]
    egl = [p for p in (EGL_PATH, "/etc/glvnd/egl_vendor.d/10_nvidia.json") if os.path.exists(p)]
    return {"vulkan_icd": bool(icd), "egl_vendor": bool(egl), "libEGL_nvidia": has("libEGL_nvidia.so.0"), "libGLX_nvidia": has("libGLX_nvidia.so.0"),
            "glcore_matches_driver": has(f"libnvidia-glcore.so.{DRIVER}")}

def install_from_apt(driver):
    """Ubuntu's libnvidia-gl-<major>, but only at exactly the driver's version (a different patch level gives 'API mismatch')."""
    major = driver.split(".")[0]
    if not S.get("apt_updated"):
        sh([*SUDO, "apt-get", "update", "-qq"], check=False, echo=False); S["apt_updated"] = True
    mad = sh(["apt-cache", "madison", f"libnvidia-gl-{major}"], echo=False, check=False).out
    vers = [l.split("|")[1].strip() for l in mad.splitlines() if l.count("|") >= 2]
    exact = [v for v in vers if v.split("-")[0].split(":")[-1] == driver]
    if not exact:
        print(f"  apt: libnvidia-gl-{major} is available as {sorted(set(vers))[:3] or 'nothing'}, not as {driver}")
        return False
    env = {**os.environ, "DEBIAN_FRONTEND": "noninteractive"}
    pk = [f"libnvidia-gl-{major}={exact[0]}", f"libnvidia-common-{major}={exact[0]}"]
    if sh([*SUDO, "apt-get", "install", "-y", "-qq", "--no-install-recommends", *pk], env=env, check=False, echo=False).rc == 0:
        print(f"  apt: installed libnvidia-gl-{major} {exact[0]}"); return True
    print("  apt: install of the exact version failed"); return False

NV_GL_FILES = ["libEGL_nvidia", "libGLX_nvidia", "libGLESv1_CM_nvidia", "libGLESv2_nvidia", "libnvidia-glcore", "libnvidia-eglcore", "libnvidia-glsi",
               "libnvidia-glvkspirv", "libnvidia-rtcore", "libnvidia-gpucomp", "libnvidia-tls", "libnvidia-allocator"]
def install_from_run(driver):
    """Download NVIDIA's own installer for exactly this driver version, unpack it (no kernel module, no root install) and copy the GL libs."""
    d = f"{WORK}/nvidia"; os.makedirs(d, exist_ok=True)
    run, tgt = f"{d}/NVIDIA-Linux-x86_64-{driver}.run", f"{d}/NVIDIA-Linux-x86_64-{driver}"
    if not os.path.isdir(tgt):
        for url in (f"https://us.download.nvidia.com/tesla/{driver}/NVIDIA-Linux-x86_64-{driver}.run",
                    f"https://us.download.nvidia.com/XFree86/Linux-x86_64/{driver}/NVIDIA-Linux-x86_64-{driver}.run",
                    f"https://download.nvidia.com/XFree86/Linux-x86_64/{driver}/NVIDIA-Linux-x86_64-{driver}.run"):
            if sh(["curl", "-fL", "--retry", "2", "--connect-timeout", "20", "-o", run, url], echo=False, check=False, timeout=900).rc == 0 and os.path.getsize(run) > 10_000_000:
                print("  downloaded", url.split("/")[2], "...", f"{os.path.getsize(run) / 2**20:.0f} MB"); break
        else:
            print(f"  nvidia.com has no installer for driver {driver}"); return False
        if sh(["sh", run, "-x"], cwd=d, echo=False, check=False, timeout=600).rc != 0 or not os.path.isdir(tgt):
            print("  could not unpack the installer"); return False
    copied = []
    for base in NV_GL_FILES:
        src = f"{tgt}/{base}.so.{driver}"
        if os.path.exists(src):
            sh([*SUDO, "cp", "-f", src, f"{LIBDIR}/"], echo=False); copied.append(base)
    if "libEGL_nvidia" not in copied or "libGLX_nvidia" not in copied:
        print("  the installer did not contain libEGL_nvidia / libGLX_nvidia"); return False
    links = {"libEGL_nvidia": "libEGL_nvidia.so.0", "libGLX_nvidia": "libGLX_nvidia.so.0", "libGLESv1_CM_nvidia": "libGLESv1_CM_nvidia.so.1",
             "libGLESv2_nvidia": "libGLESv2_nvidia.so.2", "libnvidia-allocator": "libnvidia-allocator.so.1"}
    for base, link in links.items():
        if base in copied: sh([*SUDO, "ln", "-sf", f"{base}.so.{driver}", f"{LIBDIR}/{link}"], echo=False)
    write_icd_files(tgt)
    sh([*SUDO, "ldconfig"], echo=False, check=False)
    print("  copied:", ", ".join(copied)); return True

def write_icd_files(tgt):
    """The Vulkan ICD and the glvnd EGL vendor json (the installer's own copy if present, else the standard two lines)."""
    def put(path, text, src=None):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        if src and os.path.exists(src): sh([*SUDO, "cp", "-f", src, path], echo=False)
        else:
            tmp = f"{WORK}/_icd.json"; open(tmp, "w").write(text); sh([*SUDO, "cp", "-f", tmp, path], echo=False)
    put(ICD_PATH, json.dumps({"file_format_version": "1.0.0", "ICD": {"library_path": "libGLX_nvidia.so.0", "api_version": "1.3.0"}}, indent=2) + "\n", f"{tgt}/nvidia_icd.json")
    put(EGL_PATH, json.dumps({"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}, indent=2) + "\n", f"{tgt}/10_nvidia.json")

st = nv_gl_status()
print("NVIDIA graphics userspace:", st)
if all(st.values()):
    print("Colab already ships the NVIDIA EGL/Vulkan libraries for this driver: nothing to install.")
    S["nvidia_gl"] = "already present"
else:
    print(f"EGL/Vulkan pieces missing for driver {DRIVER}; installing a matching set ...")
    how = "apt (exact version)" if install_from_apt(DRIVER) else "NVIDIA .run (exact version)" if install_from_run(DRIVER) else None
    if not how:
        # last resort: Ubuntu's nearest package; it only works if the kernel driver happens to accept it
        major = DRIVER.split(".")[0]
        if sh([*SUDO, "apt-get", "install", "-y", "-qq", "--no-install-recommends", f"libnvidia-gl-{major}"], env={**os.environ, "DEBIAN_FRONTEND": "noninteractive"}, check=False, echo=False).rc == 0:
            how = f"apt libnvidia-gl-{major} (NOT the exact driver version; may fail with 'API mismatch')"
    if not how:
        print("Could not install NVIDIA graphics libraries by any route; the GPU probe below will most likely fail and say so.")
    else:
        sh([*SUDO, "ldconfig"], echo=False, check=False)
        print("installed via", how)
    S["nvidia_gl"] = how or "failed"
    st = nv_gl_status(); print("now:", st)
    if not all(st.values()): print("WARNING: still missing:", [k for k, v in st.items() if not v], "(the probe below shows whether it matters)")
for so in (f"{LIBDIR}/libEGL_nvidia.so.0", f"{LIBDIR}/libGLX_nvidia.so.0"):
    if os.path.exists(so):
        bad = [l.split()[0] for l in sh(["ldd", so], echo=False, check=False).out.splitlines() if "not found" in l]
        if bad: print(f"WARNING: {os.path.basename(so)} needs {bad}, which are missing")
save()

In [ ]:
#@title 11 · GPU probe: which Chrome flags really reach the NVIDIA GPU?
# Same launch path as the render (film/tools/harness.mjs), via `render.mjs --probe-gl`. A candidate only counts if the WebGL renderer
# string says NVIDIA: without a GPU path Chrome happily hands out SwiftShader or Mesa llvmpipe, which would render on the CPU.
CHROME = S["chrome"]
CANDIDATES = [
    {"name": "angle-vulkan", "gpu_angle": "vulkan", "flags": [], "env": {}},
    {"name": "angle-gl-egl", "gpu_angle": "gl-egl", "flags": [], "env": {}},
    {"name": "angle-vulkan, no Vulkan surface", "gpu_angle": "vulkan", "flags": ["--disable-vulkan-surface"], "env": {}},
    {"name": "angle-gl-egl, EGL_PLATFORM=device", "gpu_angle": "gl-egl", "flags": [], "env": {"EGL_PLATFORM": "device"}},
]
_extra = EXTRA_CHROME_FLAGS.split()

def gl_args(c):
    fl = c["flags"] + _extra
    return [f"--gpu-angle={c['gpu_angle']}", f"--chrome={CHROME}"] + ([f"--chrome-flags={' '.join(fl)}"] if fl else [])

def node_env(c): return {**os.environ, **c["env"]}

def is_nvidia(info):
    r = info.get("renderer") or ""
    seen = " ".join([r, info.get("vendor") or "", *(info.get("devices") or [])])
    return bool(info.get("ok")) and re.search(r"nvidia", seen, re.I) is not None and re.search(r"swiftshader|llvmpipe|softpipe|software", r, re.I) is None

def probe(c):
    r = sh(["node", "render.mjs", "--probe-gl", *gl_args(c)], cwd=FILM, env=node_env(c), echo=False, check=False, timeout=180)
    line = next((l for l in r.out.splitlines() if l.startswith("PROBE_GL ")), None)
    if not line:
        return {"ok": False, "error": "no result: " + " | ".join(r.out.strip().splitlines()[-3:])[:300]}
    try: return json.loads(line[len("PROBE_GL "):])
    except ValueError: return {"ok": False, "error": "unreadable result: " + line[:200]}

GL = None; _results = []
for c in CANDIDATES:
    info = probe(c); _results.append((c, info))
    good = is_nvidia(info)
    print(f"[{'PASS' if good else 'fail'}] {c['name']:<36} -> {info.get('renderer') or info.get('error')}")
    if good:
        GL = {**c, "renderer": info["renderer"], "vendor": info.get("vendor"), "devices": info.get("devices"), "chrome": info.get("chrome")}
        break

if not GL:
    print("\n--- diagnostics ---")
    sh("ls -l /dev/nvidia* /dev/dri 2>&1 | head -20", check=False)
    sh("ldconfig -p | grep -i -E 'libEGL|libGLX|libnvidia-(glcore|eglcore|glsi|rtcore)|libvulkan' | head -20", check=False)
    sh("for f in /etc/vulkan/icd.d/*.json /usr/share/vulkan/icd.d/*.json /usr/share/glvnd/egl_vendor.d/*.json; do [ -f $f ] && echo \"== $f\" && cat $f; done 2>/dev/null | head -40", check=False)
    for c, info in _results: print(c["name"], "->", json.dumps({k: info.get(k) for k in ("ok", "error", "renderer", "devices", "features")}))
    stop("No Chrome GPU path reached the NVIDIA GPU (see the renderer strings above). This notebook will not fall back to SwiftShader/llvmpipe:",
         "a CPU render is far too slow to be worth it.",
         f"NVIDIA graphics libraries: {S.get('nvidia_gl')}. Things to try: Runtime -> Disconnect and delete runtime, then Run all on a fresh VM;",
         "or switch GPU type (T4 / L4 / A100). If it keeps failing, send the diagnostics above (the driver version is " + str(S.get("gpu", {}).get("driver")) + ").")

S["gl"] = GL; save()
GL_ARGS = gl_args(GL); GL_ENV = node_env(GL)
print(f"\nusing {GL['name']}:\n  renderer: {GL['renderer']}\n  node render.mjs {' '.join(GL_ARGS)}" + (f"\n  env: {GL['env']}" if GL["env"] else ""))

In [ ]:
#@title 12 · Benchmark: 24 real frames on one worker (no encode)
DUR, FPS = read_project()
FRAMES = n_frames(FROM, TO, FPS, DUR)
if FRAMES <= 0: stop(f"FROM={FROM}, TO={TO} selects no frames (the film is {DUR:g} s long).")
WORKERS_EFF = int(WORKERS) if int(WORKERS) > 0 else S["workers"]

bench_from = 55 if FROM <= 55 < TO else FROM
print(f"benchmark at t = {bench_from} s (one worker, JPEG capture path, includes the first-frame warm-up) ...\n")
_gl_seen = []
def _on(line):
    if line.startswith("GL: "): _gl_seen.append(line)
    check_gl_line(line)
    return False
r = sh(["node", "render.mjs", "--bench=24", f"--from={num(bench_from)}", *GL_ARGS], cwd=FILM, env=GL_ENV, on_line=_on, timeout=1800, what="render.mjs --bench")
if not _gl_seen: stop("The benchmark never printed its `GL:` line, so the renderer could not be confirmed as NVIDIA.")
m = re.search(r"first ([\d.]+) s, then ([\d.]+) s/frame over (\d+) frames\s+\(median ([\d.]+)", r.out)
if not m: stop("Could not read the benchmark result (expected a line starting with `bench:`). Output above.")
first, spf, nb, med = map(float, m.groups())
est = spf * FRAMES / WORKERS_EFF
S["bench"] = {"at_s": bench_from, "first_frame_s": first, "s_per_frame": spf, "median_s_per_frame": med, "frames": int(nb), "gl_line": _gl_seen[0]}
S["est_total_s"] = est; save()
print(f"\n==> {spf:.2f} s/frame on one worker (median {med:.2f}, first frame {first:.1f} s)")
print(f"    {FRAMES} frames with {WORKERS_EFF} workers: about {fmt_dur(est)} if the workers scale linearly,"
      f" about {fmt_dur(est / 0.7)} if they only reach 70% (they share one GPU).")
print("    (a guide only: this is one stretch of the film, and every new shot pays for its cached layers once per worker)")
if spf > 20: print("WARNING: that is very slow for a GPU; check the renderer line above.")

In [ ]:
#@title 13 · Drive persistence: keep the chunks in MyDrive so a re-run after a disconnect resumes
DUR, FPS = read_project()
CHUNKS_LOCAL = f"{FILM}/out/chunks/1920x1080_{FPS}fps"      # where render.mjs --video keeps its resumable 1 s chunks
S["chunks_dir"] = CHUNKS_LOCAL
if not (USE_DRIVE and os.path.isdir(DRIVE_ROOT)):
    print("Drive persistence off (USE_DRIVE is False or Drive is not mounted): chunks stay in", CHUNKS_LOCAL)
else:
    target = f"{DRIVE_ROOT}/disease_called_ai/chunks_{S['sha'][:7]}"
    os.makedirs(target, exist_ok=True)
    try:        # can we really write, rename and delete there? (ffmpeg writes *.part.mp4, then render.mjs renames it)
        t1, t2 = f"{target}/.write_test", f"{target}/.write_test2"
        open(t1, "w").write("x"); os.rename(t1, t2); os.remove(t2); writable = True
    except OSError as e:
        writable = False; print("Drive folder is not writable:", e)
    if not writable:
        print("Continuing with local chunks only.")
    else:
        os.makedirs(os.path.dirname(CHUNKS_LOCAL), exist_ok=True)
        if os.path.islink(CHUNKS_LOCAL):
            if os.path.realpath(CHUNKS_LOCAL) != os.path.realpath(target):
                os.remove(CHUNKS_LOCAL); os.symlink(target, CHUNKS_LOCAL)
        else:
            if os.path.isdir(CHUNKS_LOCAL):          # chunks rendered earlier on this VM: move them into Drive, then link
                moved = 0
                for f in os.listdir(CHUNKS_LOCAL):
                    if f.endswith(".part.mp4"): continue
                    if not os.path.exists(f"{target}/{f}"): shutil.move(f"{CHUNKS_LOCAL}/{f}", f"{target}/{f}"); moved += 1
                shutil.rmtree(CHUNKS_LOCAL)
                if moved: print(f"moved {moved} existing local chunks into Drive")
            os.symlink(target, CHUNKS_LOCAL)
        have = len([f for f in os.listdir(target) if re.match(r"c\d+-\d+\.mp4$", f)])
        print(f"{CHUNKS_LOCAL}\n  -> {target}\n{have} finished chunks already there" + (f" (about {have} s of video will be skipped)" if have else " (fresh render)"))
        S["chunks_in_drive"] = target
save()

In [ ]:
#@title 14 · Full render (resumable) + MP4 check
import datetime
DUR, FPS = read_project()
FRAMES = n_frames(FROM, TO, FPS, DUR)
WORKERS_EFF = int(WORKERS) if int(WORKERS) > 0 else S["workers"]
SHORT = S["short"]
full = FROM <= 0 and TO >= DUR
SUFFIX = "" if full else f"_{num(FROM)}-{num(TO)}s"
VIDEO_NAME, LOG_NAME = f"video_{SHORT}{SUFFIX}.mp4", f"render_log{SUFFIX}.json"
VIDEO = f"{OUT_DIR}/{VIDEO_NAME}"
GL = S["gl"]
GL_ARGS = [f"--gpu-angle={GL['gpu_angle']}", f"--chrome={S['chrome']}"] + ([f"--chrome-flags={' '.join(GL['flags'] + EXTRA_CHROME_FLAGS.split())}"] if (GL["flags"] or EXTRA_CHROME_FLAGS.split()) else [])
GL_ENV = {**os.environ, **GL["env"]}

def probe_mp4(path, fps, expected_frames):
    """ffprobe the finished MP4: codec, size, fps, frame count, duration. Returns (facts, problems)."""
    p = subprocess.run(["ffprobe", "-v", "error", "-count_packets", "-select_streams", "v:0", "-show_entries",
                        "stream=codec_name,profile,width,height,pix_fmt,r_frame_rate,avg_frame_rate,nb_frames,nb_read_packets",
                        "-show_entries", "format=duration,size", "-of", "json", path], capture_output=True, text=True)
    if p.returncode != 0: return {}, [f"ffprobe could not read the file: {p.stderr.strip()[:200]}"]
    j = json.loads(p.stdout)
    st, fm = (j.get("streams") or [{}])[0], j.get("format", {})
    a, _, b = st.get("avg_frame_rate", "0/1").partition("/")
    got_fps = float(a) / float(b or 1) if float(b or 1) else 0.0
    frames = int(st.get("nb_read_packets") or st.get("nb_frames") or 0)
    dur = float(fm.get("duration") or 0)
    facts = {"codec": st.get("codec_name"), "profile": st.get("profile"), "size": f"{st.get('width')}x{st.get('height')}", "pix_fmt": st.get("pix_fmt"),
             "fps": round(got_fps, 3), "frames": frames, "nb_frames_header": st.get("nb_frames"), "duration_s": round(dur, 3), "bytes": int(fm.get("size") or 0)}
    bad = []
    if st.get("codec_name") != "h264": bad.append(f"codec is {st.get('codec_name')}, expected h264")
    if (st.get("width"), st.get("height")) != (1920, 1080): bad.append(f"size is {facts['size']}, expected 1920x1080")
    if st.get("pix_fmt") != "yuv420p": bad.append(f"pix_fmt is {st.get('pix_fmt')}, expected yuv420p")
    if abs(got_fps - fps) > 0.01: bad.append(f"fps is {got_fps:.3f}, expected {fps}")
    if frames != expected_frames: bad.append(f"{frames} frames, expected {expected_frames}")
    if st.get("nb_frames") not in (None, "N/A", str(expected_frames)): bad.append(f"container says {st.get('nb_frames')} frames, expected {expected_frames}")
    if abs(dur - expected_frames / fps) > 1.5 / fps: bad.append(f"duration {dur:.3f} s, expected {expected_frames / fps:.3f} s")
    return facts, bad

S["verified"] = False; save()          # set again only once the MP4 below has been checked
print(f"{FRAMES} frames ({num(FROM)}-{num(TO)} s at {FPS} fps), {WORKERS_EFF} workers, CRF {CRF}, GL {GL['name']} -> {VIDEO_NAME}\n")
cmd = ["node", "render.mjs", "--video", f"--from={num(FROM)}", f"--to={num(TO)}", f"--workers={WORKERS_EFF}", f"--crf={int(CRF)}", "--no-audio", f"--out={VIDEO}", *GL_ARGS]
LOGF = f"{WORK}/render_stdout.log"; open(LOGF, "w").close()
_prog = re.compile(r"^frame (\d+)/(\d+)\s+([\d.]+) s/frame wall \(([\d.]+) per worker\)\s+eta (.+)$")
_state = {"last": None, "line_open": False, "gl": None}
def _on(line):
    if line.startswith("GL: "): _state["gl"] = line
    check_gl_line(line)
    m = _prog.match(line)
    if m:
        d, t, wall, perw, eta = m.groups(); _state["last"] = (int(d), int(t), float(wall), float(perw))
        print(f"\r  rendering {d}/{t} frames  ({int(d) / int(t) * 100:.1f}%)   {wall} s/frame overall   eta {eta}        ", end="", flush=True)
        _state["line_open"] = True
        return True
    if _state["line_open"]: print(); _state["line_open"] = False
    return False

t0, attempt, rc = time.time(), 0, 1
while rc != 0:
    attempt += 1
    rc = sh(cmd, cwd=FILM, env=GL_ENV, on_line=_on, logfile=LOGF, check=False, what="render.mjs --video").rc
    if _state["line_open"]: print(); _state["line_open"] = False
    if rc != 0:
        sweep_chrome()
        print(f"\nrender.mjs exited with {rc} (attempt {attempt}/{RETRIES + 1}). Finished chunks are kept.")
        if attempt > RETRIES:
            stop("The render failed repeatedly; the last output is above and in " + LOGF + ".",
                 "Run this cell again to resume from the finished chunks (try a smaller WORKERS if Chrome ran out of memory).")
wall = time.time() - t0
if not _state["gl"]: stop("The render never printed its `GL:` line, so it was never confirmed to run on the NVIDIA GPU.")
if not os.path.exists(VIDEO): stop(f"render.mjs finished but {VIDEO} does not exist.")

facts, problems = probe_mp4(VIDEO, FPS, FRAMES)
print("\nffprobe:", json.dumps(facts))
if problems: stop("The MP4 does not look right:", *problems, "(the chunks are still there; fix the cause and run this cell again)")
print(f"MP4 verified: {facts['frames']} frames, {facts['duration_s']} s, {facts['fps']} fps, {facts['size']}, {facts['bytes'] / 2**20:.0f} MB")

last = _state["last"]
LOG = {
    "tag": S["tag"], "sha": S["sha"], "branch": S["branch"], "commit_subject": S["subject"],
    "gpu": S["gpu"]["name"], "gpu_driver": S["gpu"]["driver"], "gpu_memory_mib": S["gpu"]["memory_mib"],
    "gl_renderer": GL["renderer"], "gl_candidate": GL["name"], "gl_line": _state["gl"],
    "flags": {"render_mjs": cmd[2:], "chrome_flags": GL["flags"] + EXTRA_CHROME_FLAGS.split(), "env": GL["env"]},
    "workers": WORKERS_EFF, "crf": int(CRF), "from_s": FROM, "to_s": TO, "fps": FPS, "frames": FRAMES,
    "s_per_frame_wall": (last[2] if last else None), "s_per_frame_per_worker": (last[3] if last else None),
    "bench_s_per_frame_one_worker": (S.get("bench") or {}).get("s_per_frame"),
    "wall_seconds_this_run": round(wall, 1), "attempts": attempt, "chunks_in_drive": S.get("chunks_in_drive"),
    "chrome": S.get("chrome_version"), "node": S.get("node"), "ffmpeg": S.get("ffmpeg"),
    "vcpu": S["gpu"]["vcpu"], "ram_gb": S["gpu"]["ram_gb"], "nvidia_gl_libs": S.get("nvidia_gl"),
    "ffprobe": facts, "video_file": VIDEO_NAME, "audio": "none (muxed later)",
    "finished_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
}
LOG["video_sha256"] = sha256_file(VIDEO)
with open(f"{OUT_DIR}/{LOG_NAME}", "w") as f: json.dump(LOG, f, indent=2)
S.update(video=VIDEO, video_name=VIDEO_NAME, log_name=LOG_NAME, verified=True); save()
print(f"\nrender took {fmt_dur(wall)} wall" + (f", {last[2]:.2f} s/frame overall with {WORKERS_EFF} workers" if last else " (all chunks were already done)") + f"  | log: {OUT_DIR}/{LOG_NAME}")

In [ ]:
#@title 15 · Upload the MP4 and render_log.json to a GitHub release
if not S.get("verified") or not os.path.exists(S.get("video", "")):
    stop("There is no verified MP4 yet: run the render cell first.")
VIDEO, VIDEO_NAME, LOG_NAME, TAG = S["video"], S["video_name"], S["log_name"], S["tag"]
LOG_PATH = f"{OUT_DIR}/{LOG_NAME}"
size = os.path.getsize(VIDEO)
def plan_parts(name, size, limit, part):
    """One asset if it fits under GitHub's limit, else raw byte parts name.part001, .part002, ... (reassemble with cat)."""
    if size < limit: return [{"name": name, "offset": 0, "length": size}]
    return [{"name": f"{name}.part{i + 1:03d}", "offset": o, "length": min(part, size - o)} for i, o in enumerate(range(0, size, part))]
parts = plan_parts(VIDEO_NAME, size, GH_LIMIT, PART)
for p in parts: p["sha256"] = sha256_file(VIDEO, p["offset"], p["length"])
log = json.load(open(LOG_PATH))
log["release"] = {"repo": RELEASE_REPO, "tag": TAG}
log["assets"] = [{"name": p["name"], "bytes": p["length"], "sha256": p["sha256"]} for p in parts] + [{"name": LOG_NAME}]
if len(parts) > 1:
    log["reassemble"] = f"cat {VIDEO_NAME}.part* > {VIDEO_NAME}   # then check sha256 = video_sha256"
json.dump(log, open(LOG_PATH, "w"), indent=2)
print(f"{VIDEO_NAME}: {size / 2**20:.0f} MB" + (f" -> {len(parts)} parts (over GitHub's 2 GiB limit)" if len(parts) > 1 else "") + f"  | release {RELEASE_REPO} @ {TAG}")


class SliceReader:
    """A window [offset, offset+length) of a file, read by requests as the upload body, with a progress line."""
    def __init__(self, path, offset, length, label):
        self.f = open(path, "rb"); self.f.seek(offset)
        self.length, self.pos, self.label, self.t0, self.shown = length, 0, label, time.time(), 0.0
    def __len__(self): return self.length
    def tell(self): return self.pos
    def close(self): self.f.close()
    def read(self, n=-1):
        left = self.length - self.pos
        if left <= 0: return b""
        b = self.f.read(left if n is None or n < 0 else min(n, left)); self.pos += len(b)
        now = time.time()
        if self.length > 4 * 2**20 and (now - self.shown > 2 or self.pos >= self.length):
            self.shown = now
            print(f"\r  uploading {self.label}  {self.pos / self.length * 100:5.1f}%  {self.pos / 2**20:.0f}/{self.length / 2**20:.0f} MB  {self.pos / 2**20 / max(now - self.t0, 1e-6):.0f} MB/s   ", end="", flush=True)
        return b


def find_release(token):
    r = gh("GET", f"/repos/{RELEASE_REPO}/releases/tags/{urllib.parse.quote(TAG)}", token)
    if r.status_code == 200: return r.json()
    for page in range(1, 11):                                  # a draft is not reachable through /releases/tags/
        r = gh("GET", f"/repos/{RELEASE_REPO}/releases?per_page=100&page={page}", token)
        if r.status_code != 200 or not r.json(): break
        for rel in r.json():
            if rel.get("tag_name") == TAG: return rel
    return None

def get_or_create_release(token):
    rel = find_release(token)
    if rel: print(f"reusing release {rel['html_url']}"); return rel
    body = (f"Colab GPU render of `{S['sha']}` (branch `{S['branch']}`)\n\n{S['subject']}\n\n"
            f"- GPU: {S['gpu']['name']}, driver {S['gpu']['driver']}\n- WebGL renderer: `{S['gl']['renderer']}`\n"
            f"- {FPS} fps, CRF {int(CRF)}, {num(FROM)}-{num(TO)} s, no audio (mux master.wav on your side)\n\n"
            f"Download: `tools/fetch_colab_render.sh {TAG}`")
    r = gh("POST", f"/repos/{RELEASE_REPO}/releases", token, json={"tag_name": TAG, "name": TAG, "body": body, "draft": bool(DRAFT_RELEASE), "prerelease": False})
    if r.status_code == 422:                                   # somebody (or an earlier run) created it a moment ago
        rel = find_release(token)
        if rel: return rel
    if r.status_code not in (200, 201):
        stop(f"Could not create release {TAG} in {RELEASE_REPO} (HTTP {r.status_code}): {redact(r.text)[:300]}",
             "The PAT needs Contents: Read and write on that repository.")
    print(f"created release {r.json()['html_url']}"); return r.json()

def delete_same_name(token, rel, name):
    r = gh("GET", f"/repos/{RELEASE_REPO}/releases/{rel['id']}/assets?per_page=100", token)
    for a in (r.json() if r.status_code == 200 else []):
        if a["name"] == name:
            d = gh("DELETE", f"/repos/{RELEASE_REPO}/releases/assets/{a['id']}", token)
            print(f"  replaced existing asset {name} (deleted id {a['id']}, HTTP {d.status_code})")

def upload_asset(token, rel, name, path, offset, length, ctype):
    import requests
    delete_same_name(token, rel, name)
    url = f"{GITHUB_UPLOADS}/repos/{RELEASE_REPO}/releases/{rel['id']}/assets?name={urllib.parse.quote(name)}"
    for attempt in range(1, 4):
        rd = SliceReader(path, offset, length, name)
        try:
            r = requests.post(url, data=rd, timeout=(30, 1800), headers={"Authorization": f"Bearer {token}", "Accept": "application/vnd.github+json",
                              "X-GitHub-Api-Version": "2022-11-28", "Content-Type": ctype, "User-Agent": "mv-colab-render"})
        except requests.RequestException as e:
            print(f"\n  upload of {name} failed ({type(e).__name__}: {redact(e)[:160]}); attempt {attempt}/3")
            r = None
        finally:
            rd.close()
        if length > 4 * 2**20: print()
        if r is not None and r.status_code in (200, 201):
            j = r.json()
            if j.get("size") == length and j.get("state", "uploaded") == "uploaded": return j
            print(f"  GitHub reports size {j.get('size')} / state {j.get('state')}, expected {length}; retrying")
        elif r is not None:
            print(f"  HTTP {r.status_code}: {redact(r.text)[:200]}")
            if r.status_code in (401, 403, 404): stop("GitHub refused the upload: the PAT needs Contents: Read and write on " + RELEASE_REPO + ".")
        time.sleep(5 * attempt); delete_same_name(token, rel, name)
    stop(f"Could not upload {name} after 3 attempts.", "The video is still on this VM and in Drive's chunks: run this cell again.")

items = [(p["name"], VIDEO, p["offset"], p["length"], "video/mp4" if len(parts) == 1 else "application/octet-stream") for p in parts]
items.append((LOG_NAME, LOG_PATH, 0, os.path.getsize(LOG_PATH), "application/json"))

if DRY_RUN:
    print("\nDRY_RUN: would create or reuse the release and upload (replacing same-named assets):")
    for n, _, off, ln, ct in items: print(f"  {n}  {ln / 2**20:.1f} MB  {ct}")
    print(f"release URL would be https://github.com/{RELEASE_REPO}/releases/tag/{TAG}")
else:
    token = get_token()
    rel = get_or_create_release(token)
    for n, path, off, ln, ct in items:
        j = upload_asset(token, rel, n, path, off, ln, ct)
        print(f"  uploaded {n}  ({j['size'] / 2**20:.1f} MB)")
    S["release_url"] = rel["html_url"]; save()
    print(f"\nDONE.  Release: {rel['html_url']}")
    print(f"On your machine:  tools/fetch_colab_render.sh {TAG}")